In [0]:
import json
import mlflow

import math
from typing import List, Dict, Any


def clamp(x, lo=0.0, hi=1.0):
    return max(lo, min(hi, x))


def compute_desert_scores(
    recommended_results: List[Dict[str, Any]],
    radius_km: float
) -> Dict[str, float]:
    if not recommended_results:
        return {
            "coverage_score": 0.0,
            "desert_score": 1.0,
            "confidence_score": 0.15
        }

    distances = []
    trusts = []

    for r in recommended_results:
        if r.get("distance_km") is not None:
            distances.append(float(r["distance_km"]))
        trusts.append(float(r.get("final_score", 0.0)))

    min_distance = min(distances) if distances else radius_km
    avg_trust = sum(trusts) / len(trusts) if trusts else 0.0
    facility_count_factor = clamp(len(recommended_results) / 5.0)

    distance_factor = 1.0 - clamp(min_distance / radius_km) if radius_km > 0 else 0.0
    trust_factor = clamp(avg_trust)

    coverage_score = (
        0.40 * facility_count_factor +
        0.30 * distance_factor +
        0.30 * trust_factor
    )

    desert_score = 1.0 - coverage_score

    confidence_score = clamp(
        0.5 * facility_count_factor +
        0.5 * trust_factor
    )

    return {
        "coverage_score": round(coverage_score, 4),
        "desert_score": round(desert_score, 4),
        "confidence_score": round(confidence_score, 4)
    }


def build_area_reasoning(
    medical_need: str,
    radius_km: float,
    candidate_count: int,
    recommended_count: int,
    score_bundle: Dict[str, float]
) -> Dict[str, Any]:
    steps = [
        f"Interpreted target capability as '{medical_need}'",
        f"Searched for facilities within {radius_km} km",
        f"Evaluated candidate facilities for recommendation-worthiness",
        f"Computed coverage and desert scores from count, distance, and trust",
        f"Flagged area as desert when coverage is too weak"
    ]

    summary = (
        f"For medical need '{medical_need}', {candidate_count} nearby candidates were checked, "
        f"{recommended_count} were recommendable, resulting in coverage_score="
        f"{score_bundle['coverage_score']} and desert_score={score_bundle['desert_score']}."
    )

    return {
        "summary": summary,
        "reasoning_steps": steps
    }


@mlflow.trace
def detect_medical_deserts_for_points(
    points: List[Dict[str, Any]],
    medical_need: str,
    table_name: str = "facilities_scored",
    search_radius_km: float = 50.0,
    max_candidates: int = 50,
    top_k: int = 20,
    min_final_score: float = 0.45,
    desert_threshold: float = 0.60,
) -> Dict[str, Any]:
    outputs = []

    for point in points:
        lat = float(point["lat"])
        lon = float(point["lon"])
        region_id = point.get("region_id") or f"{lat:.4f}_{lon:.4f}"
        postal_code = point.get("postal_code")

        query_plan = {
            "medical_need": medical_need,
            "urgency": "high",
            "trust_threshold": min_final_score,
            "distance_preference": "nearest",
            "origin": {"lat": lat, "lon": lon},
            "constraints": [
                {"field": "distance_km", "operator": "<=", "value": search_radius_km}
            ],
            "reasoning_steps": [
                "Use geographic origin from region centroid",
                "Find nearby facilities for target capability",
                "Recommend only high-confidence facilities",
                "Estimate whether this region is under-served"
            ]
        }

        candidate_pool = retrieve_candidate_pool(
            query_plan=query_plan,
            table_name=table_name,
            max_candidates=max_candidates
        )

        user_query = f"{medical_need} near lat {lat}, lon {lon}"

        ranked_results = rerank_candidates_with_llm(
            user_query=user_query,
            query_plan=query_plan,
            candidates=candidate_pool,
            top_k=top_k
        )

        recommended_results = [
            r for r in ranked_results
            if bool((r.get("judge") or {}).get("should_recommend", False))
            and float(r.get("final_score", 0.0)) >= float(min_final_score)
        ]

        score_bundle = compute_desert_scores(
            recommended_results=recommended_results,
            radius_km=search_radius_km
        )

        best_distance = None
        if recommended_results:
            dvals = [float(r["distance_km"]) for r in recommended_results if r.get("distance_km") is not None]
            best_distance = min(dvals) if dvals else None

        avg_trust = None
        if recommended_results:
            avg_trust = sum(float(r.get("final_score", 0.0)) for r in recommended_results) / len(recommended_results)

        is_medical_desert = float(score_bundle["desert_score"]) >= float(desert_threshold)

        area_reasoning = build_area_reasoning(
            medical_need=medical_need,
            radius_km=search_radius_km,
            candidate_count=len(candidate_pool),
            recommended_count=len(recommended_results),
            score_bundle=score_bundle
        )

        outputs.append({
            "region_id": region_id,
            "postal_code": postal_code,
            "lat": lat,
            "lon": lon,
            "medical_need": medical_need,
            "search_radius_km": search_radius_km,
            "candidate_pool_size": len(candidate_pool),
            "recommended_facility_count": len(recommended_results),
            "best_facility_distance_km": round(best_distance, 3) if best_distance is not None else None,
            "avg_recommended_trust": round(avg_trust, 4) if avg_trust is not None else None,
            "coverage_score": score_bundle["coverage_score"],
            "desert_score": score_bundle["desert_score"],
            "confidence_score": score_bundle["confidence_score"],
            "is_medical_desert": is_medical_desert,
            "query_reasoning": query_plan,
            "area_reasoning": area_reasoning,
            "top_recommendations": recommended_results[:5]
        })

    outputs = sorted(outputs, key=lambda x: x["desert_score"], reverse=True)

    return {
        "ok": True,
        "medical_need": medical_need,
        "search_radius_km": search_radius_km,
        "desert_threshold": desert_threshold,
        "region_count": len(outputs),
        "results": outputs
    }

In [0]:
dbutils.widgets.text("medical_need", "emergency_surgery")
dbutils.widgets.text("search_radius_km", "50")
dbutils.widgets.text("desert_threshold", "0.60")

medical_need = dbutils.widgets.get("medical_need").strip()
search_radius_km = float(dbutils.widgets.get("search_radius_km").strip() or "50")
desert_threshold = float(dbutils.widgets.get("desert_threshold").strip() or "0.60")

# Beispiel: points muss vorher geladen werden, z. B. aus PIN-Code-Centroids oder Grid-Zellen
# points = [{"region_id": "110001", "postal_code": "110001", "lat": 28.6353, "lon": 77.2250}, ...]

response = detect_medical_deserts_for_points(
    points=points,
    medical_need=medical_need,
    table_name="facilities_scored",
    search_radius_km=search_radius_km,
    max_candidates=50,
    top_k=20,
    min_final_score=0.45,
    desert_threshold=desert_threshold
)

dbutils.notebook.exit(json.dumps(response, ensure_ascii=False))

In [0]:
import json

dbutils.widgets.text("medical_need", "emergency_surgery")
medical_need = dbutils.widgets.get("medical_need").strip()

table_name_final = f"medical_desert_results_{medical_need.replace(' ', '_')}"

try:
    rows = spark.sql(f"SELECT results FROM {table_name_final} LIMIT 1").collect()

    if not rows:
        dbutils.notebook.exit(json.dumps({
            "ok": False,
            "error": f"No stored results found for {medical_need}"
        }, ensure_ascii=False))

    result_json = rows[0]["results"]
    dbutils.notebook.exit(result_json)

except Exception as e:
    dbutils.notebook.exit(json.dumps({
        "ok": False,
        "error": str(e)
    }, ensure_ascii=False))

In [0]:
# 3. UNVERÄNDERT - Funktioniert
result = process_and_store_medical_deserts(
    medical_need="emergency_surgery",
    source_table="facilities_scored",
    target_table_prefix="medical_desert_results",
    search_radius_km=50.0,
    max_candidates=50,
    top_k=20,
    min_final_score=0.45,
    desert_threshold=0.60
)

print(result)

In [0]:
# 2. GEFIXT - Korrekte Spaltennamen aus Dataset: lat, lon, pincode
import json
from pyspark.sql import functions as F

@mlflow.trace
def process_and_store_medical_deserts(
    medical_need: str,
    source_table: str = "facilities_scored",
    target_table_prefix: str = "medical_desert_results",
    search_radius_km: float = 50.0,
    max_candidates: int = 50,
    top_k: int = 20,
    min_final_score: float = 0.45,
    desert_threshold: float = 0.60,
) -> Dict[str, Any]:

    pin_points_df = (
        spark.table("facilities_scored")
        .filter(F.col("pincode").isNotNull())
        .filter(F.col("lat").isNotNull())
        .filter(F.col("lon").isNotNull())
        .withColumn("postal_code", F.trim(F.col("pincode").cast("string")))
        .withColumn("lat", F.col("lat").cast("double"))
        .withColumn("lon", F.col("lon").cast("double"))
        .groupBy("postal_code")
        .agg(
            F.avg("lat").alias("lat"),
            F.avg("lon").alias("lon"),
            F.count("*").alias("facility_count_in_pin")
        )
        .filter(F.length("postal_code") > 0)
    )

    points = [
        {
            "region_id": row["postal_code"],
            "postal_code": row["postal_code"],
            "lat": float(row["lat"]),
            "lon": float(row["lon"]),
            "facility_count_in_pin": int(row["facility_count_in_pin"])
        }
        for row in pin_points_df.collect()
    ]

    response = detect_medical_deserts_for_points(
        points=points,
        medical_need=medical_need,
        table_name=source_table,
        search_radius_km=search_radius_km,
        max_candidates=max_candidates,
        top_k=top_k,
        min_final_score=min_final_score,
        desert_threshold=desert_threshold
    )

    target_table = f"{target_table_prefix}_{medical_need.replace(' ', '_')}"

    df_final = spark.createDataFrame([{
        "medical_need": medical_need,
        "source_table": source_table,
        "search_radius_km": search_radius_km,
        "max_candidates": max_candidates,
        "top_k": top_k,
        "min_final_score": min_final_score,
        "desert_threshold": desert_threshold,
        "results": json.dumps(response, ensure_ascii=False)
    }])

    (
        df_final.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    return {
        "ok": True,
        "medical_need": medical_need,
        "target_table": target_table,
        "region_count": response.get("region_count", 0)
    }

In [0]:
# 1. UNVERÄNDERT - Funktioniert
import json
import mlflow
import math
from typing import List, Dict, Any

def clamp(x, lo=0.0, hi=1.0):
    return max(lo, min(hi, x))

def compute_desert_scores(
    recommended_results: List[Dict[str, Any]],
    radius_km: float
) -> Dict[str, float]:
    if not recommended_results:
        return {
            "coverage_score": 0.0,
            "desert_score": 1.0,
            "confidence_score": 0.15
        }

    distances = []
    trusts = []

    for r in recommended_results:
        if r.get("distance_km") is not None:
            distances.append(float(r["distance_km"]))
        trusts.append(float(r.get("final_score", 0.0)))

    min_distance = min(distances) if distances else radius_km
    avg_trust = sum(trusts) / len(trusts) if trusts else 0.0
    facility_count_factor = clamp(len(recommended_results) / 5.0)

    distance_factor = 1.0 - clamp(min_distance / radius_km) if radius_km > 0 else 0.0
    trust_factor = clamp(avg_trust)

    coverage_score = (
        0.40 * facility_count_factor +
        0.30 * distance_factor +
        0.30 * trust_factor
    )

    desert_score = 1.0 - coverage_score

    confidence_score = clamp(
        0.5 * facility_count_factor +
        0.5 * trust_factor
    )

    return {
        "coverage_score": round(coverage_score, 4),
        "desert_score": round(desert_score, 4),
        "confidence_score": round(confidence_score, 4)
    }

def build_area_reasoning(
    medical_need: str,
    radius_km: float,
    candidate_count: int,
    recommended_count: int,
    score_bundle: Dict[str, float]
) -> Dict[str, Any]:
    steps = [
        f"Interpreted target capability as '{medical_need}'",
        f"Searched for facilities within {radius_km} km",
        f"Evaluated candidate facilities for recommendation-worthiness",
        f"Computed coverage and desert scores from count, distance, and trust",
        f"Flagged area as desert when coverage is too weak"
    ]

    summary = (
        f"For medical need '{medical_need}', {candidate_count} nearby candidates were checked, "
        f"{recommended_count} were recommendable, resulting in coverage_score="
        f"{score_bundle['coverage_score']} and desert_score={score_bundle['desert_score']}."
    )

    return {
        "summary": summary,
        "reasoning_steps": steps
    }

@mlflow.trace
def detect_medical_deserts_for_points(
    points: List[Dict[str, Any]],
    medical_need: str,
    table_name: str = "facilities_scored",
    search_radius_km: float = 50.0,
    max_candidates: int = 50,
    top_k: int = 20,
    min_final_score: float = 0.45,
    desert_threshold: float = 0.60,
) -> Dict[str, Any]:
    outputs = []

    for point in points:
        lat = float(point["lat"])
        lon = float(point["lon"])
        region_id = point.get("region_id") or f"{lat:.4f}_{lon:.4f}"
        postal_code = point.get("postal_code")

        query_plan = {
            "medical_need": medical_need,
            "urgency": "high",
            "trust_threshold": min_final_score,
            "distance_preference": "nearest",
            "origin": {"lat": lat, "lon": lon},
            "constraints": [
                {"field": "distance_km", "operator": "<=", "value": search_radius_km}
            ],
            "reasoning_steps": [
                "Use geographic origin from region centroid",
                "Find nearby facilities for target capability",
                "Recommend only high-confidence facilities",
                "Estimate whether this region is under-served"
            ]
        }

        candidate_pool = retrieve_candidate_pool(
            query_plan=query_plan,
            table_name=table_name,
            max_candidates=max_candidates
        )

        user_query = f"{medical_need} near lat {lat}, lon {lon}"

        ranked_results = rerank_candidates_with_llm(
            user_query=user_query,
            query_plan=query_plan,
            candidates=candidate_pool,
            top_k=top_k
        )

        recommended_results = [
            r for r in ranked_results
            if bool((r.get("judge") or {}).get("should_recommend", False))
            and float(r.get("final_score", 0.0)) >= float(min_final_score)
        ]

        score_bundle = compute_desert_scores(
            recommended_results=recommended_results,
            radius_km=search_radius_km
        )

        best_distance = None
        if recommended_results:
            dvals = [float(r["distance_km"]) for r in recommended_results if r.get("distance_km") is not None]
            best_distance = min(dvals) if dvals else None

        avg_trust = None
        if recommended_results:
            avg_trust = sum(float(r.get("final_score", 0.0)) for r in recommended_results) / len(recommended_results)

        is_medical_desert = float(score_bundle["desert_score"]) >= float(desert_threshold)

        area_reasoning = build_area_reasoning(
            medical_need=medical_need,
            radius_km=search_radius_km,
            candidate_count=len(candidate_pool),
            recommended_count=len(recommended_results),
            score_bundle=score_bundle
        )

        outputs.append({
            "region_id": region_id,
            "postal_code": postal_code,
            "lat": lat,
            "lon": lon,
            "medical_need": medical_need,
            "search_radius_km": search_radius_km,
            "candidate_pool_size": len(candidate_pool),
            "recommended_facility_count": len(recommended_results),
            "best_facility_distance_km": round(best_distance, 3) if best_distance is not None else None,
            "avg_recommended_trust": round(avg_trust, 4) if avg_trust is not None else None,
            "coverage_score": score_bundle["coverage_score"],
            "desert_score": score_bundle["desert_score"],
            "confidence_score": score_bundle["confidence_score"],
            "is_medical_desert": is_medical_desert,
            "query_reasoning": query_plan,
            "area_reasoning": area_reasoning,
            "top_recommendations": recommended_results[:5]
        })

    outputs = sorted(outputs, key=lambda x: x["desert_score"], reverse=True)

    return {
        "ok": True,
        "medical_need": medical_need,
        "source_table": table_name,
        "search_radius_km": search_radius_km,
        "desert_threshold": desert_threshold,
        "region_count": len(outputs),
        "results": outputs
    }